In [ ]:
import sys
from pathlib import Path
import os
import numpy as np
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
import time
from group_lasso import GroupLasso
from sklearn.linear_model import Lasso

# Get the directory where this notebook is located
notebook_dir = Path(os.getcwd())
project_root = notebook_dir.parent  # Go up one level to project root
src_path = project_root / 'src'

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from cs_priors.plotting.plotting import (
    plot_overview,
    wrapper_plot_geometry,
    plot_equation,
    plot_matrices,
)
from cs_priors.simulation.mixing_model import run_simulation, just_YAX_from_simulation
from correct_sparsity import heatmap_average_wrong_sources, vector_wrong_predictions, noise_threshold
from cs_priors.solvers.vectorized_sparse_prior import sparse_prior_solution
from cs_priors.solvers.complex_lasso import complex_lasso
from cs_priors.solvers.moe_group_lasso import frequency_group_lasso
from cs_priors.solvers.real_augmented import to_real_augmented, from_real_augmented


# 1. Svikter Group LASSO for enkle eksempler?

In [ ]:
X_true = np.zeros(100)
X_true[[10, 11, 50]] = 1
groups = np.arange(len(X_true))  # Each source is its own group 
num_sources = len(X_true)
num_mics = int(0.3 * num_sources)

A = np.random.randn(num_mics, num_sources) #+ 1j * np.random.randn(num_mics, num_sources)
Y = A @ X_true
X0 = np.linalg.pinv(A) @ Y

# try to match alpha to group_reg
alpha = 1e-4
group_reg = num_mics * alpha

# They can not be identical as sklearn uses coordinate descent, group_lasso uses proximal gradient



max_iter = 10000


In [ ]:

model_lasso = Lasso(alpha=alpha, max_iter=max_iter)
model_lasso.fit(A, Y)
X_lasso = model_lasso.coef_.reshape(-1, 1)

model_gl = GroupLasso(
    groups=groups, group_reg=group_reg, n_iter=max_iter, supress_warning=True
)
model_gl.fit(A, Y)
X_gl = model_gl.coef_.reshape(-1, 1)

print(f"{num_sources} sources, {num_mics} microphones")
plot_matrices([X_true, X0, X_lasso, X_gl], titles=["True X", "Pseudoinverse X0", "Lasso", "Group Lasso"])


# 1.5 Hvilken optimeringsmetode bruker Group LASSO?

In [ ]:
import inspect
from group_lasso._group_lasso import BaseGroupLasso
print(inspect.getsource(BaseGroupLasso._minimise_loss))

# 2. Takler Group LASSO komplekse tall skrevet som reelle?

In [ ]:
X_true = np.zeros(10, dtype=complex)
X_true[[2, 5]] = 1+1j
groups = np.arange(len(X_true))
groups = np.concatenate([groups, groups])  # Duplicate groups for real and imaginary parts
num_sources = len(X_true)
num_mics = int(0.6 * num_sources)
alpha = 1e-4
group_reg = num_mics * alpha
max_iter = 10000

A = np.random.randn(num_mics, num_sources) + 1j * np.random.randn(num_mics, num_sources)
Y = A @ X_true
X0 = np.linalg.pinv(A) @ Y
Y_real = to_real_augmented(Y)
A_real = to_real_augmented(A)

X_lasso = complex_lasso(Y, A, alpha=alpha, max_iter=max_iter)

model_gl = GroupLasso(
    groups=groups, group_reg=group_reg, n_iter=max_iter, supress_warning=True
)


X_gl = from_real_augmented(model_gl.fit(A_real, Y_real).coef_.reshape(-1, 1))

plot_matrices([X_true, X0, A, Y, X_lasso, X_gl], titles=["True X", "Pseudoinverse X0", "Mixing Matrix A", "Measurements Y", "r-Lasso X", "Group Lasso X"], show_values=False)




### 3. Hvordan påvirker gruppene løsningene for simuleringsdata?

In [ ]:
# simulation

num_sources = 100
num_mics = int(0.3 * num_sources)
s_sparse = 2
seed = 1
angle_step = 2*np.pi/num_sources

sim = run_simulation(num_sources=num_sources, num_mics=num_mics, s_sparse=s_sparse, seed=seed, angle_step=angle_step)
f = 1
Y, A, X_true = sim.Y[:,f], sim.C[:,:,f], sim.X[:,f]
X0 = np.linalg.pinv(A) @ Y
max_iter = 100000
X_lasso = complex_lasso(Y, A, alpha=alpha, max_iter=max_iter)

# no group
groups = np.arange(2*len(X_true))
model = GroupLasso(
    groups=groups, group_reg=group_reg, n_iter=max_iter, supress_warning=True
)
X_nogroup = from_real_augmented(model.fit(to_real_augmented(A), to_real_augmented(Y)).coef_.reshape(-1, 1))

# Group on complex pairs
groups = np.concatenate([np.arange(len(X_true)), np.arange(len(X_true))])  # Duplicate groups for real and imaginary parts
model = GroupLasso(
    groups=groups, group_reg=group_reg, n_iter=max_iter, supress_warning=True
)
X_cgroup = from_real_augmented(model.fit(to_real_augmented(A), to_real_augmented(Y)).coef_.reshape(-1, 1))

# Group as if solution is purely real or purely imaginary
groups = np.concatenate([np.zeros(len(X_true)), np.ones(len(X_true))])
model = GroupLasso(
    groups=groups, group_reg=group_reg, n_iter=max_iter, supress_warning=True
)
X_rigroup = from_real_augmented(model.fit(to_real_augmented(A), to_real_augmented(Y)).coef_.reshape(-1, 1))

# plotting
wrapper_plot_geometry(sim, skip_labels=True)

plot_matrices(
    [X_true, X0, X_lasso, X_nogroup, X_cgroup, X_rigroup], 
    titles=["True X", "Pseudoinverse X0", "Lasso", "Group Lasso No Grouping", "Group Lasso Complex Grouping", "Group Lasso pure Real/Imaginary Grouping"]
)



### 4. Quantify how good each grouping is

In [ ]:
def no_group_solution(Y, A):
    num_sources = A.shape[1]
    groups = np.arange(2*num_sources)  # Each real and imaginary part is its own group
    alpha = 1e-4
    group_reg = A.shape[0] * alpha
    max_iter = 20000

    model = GroupLasso(
        groups=groups, group_reg=group_reg, n_iter=max_iter, supress_warning=True
    )
    X_nogroup = from_real_augmented(model.fit(to_real_augmented(A), to_real_augmented(Y)).coef_.reshape(-1, 1))
    return X_nogroup

def c_group_solution(Y, A):
    num_sources = A.shape[1]
    groups = np.concatenate([np.arange(num_sources), np.arange(num_sources)])  # Duplicate groups for real and imaginary parts
    alpha = 1e-4
    group_reg = A.shape[0] * alpha
    max_iter = 20000

    model = GroupLasso(
        groups=groups, group_reg=group_reg, n_iter=max_iter, supress_warning=True
    )
    X_cgroup = from_real_augmented(model.fit(to_real_augmented(A), to_real_augmented(Y)).coef_.reshape(-1, 1))
    return X_cgroup

def rigroup_solution(Y, A):
    num_sources = A.shape[1]
    groups = np.concatenate([np.zeros(num_sources), np.ones(num_sources)])
    alpha = 1e-4
    group_reg = A.shape[0] * alpha
    max_iter = 20000

    model = GroupLasso(
        groups=groups, group_reg=group_reg, n_iter=max_iter, supress_warning=True
    )
    X_rigroup = from_real_augmented(model.fit(to_real_augmented(A), to_real_augmented(Y)).coef_.reshape(-1, 1))
    return X_rigroup

source_range = [20]
mic_range = [int(p * source_range[0]) for p in [0.1, 0.2, 0.7, 0.9]]
sparsity_range = [1,2,3]
seeds = 10
vmin = 0
vmax = source_range[0]//2

### No group

In [ ]:
heatmap_average_wrong_sources(no_group_solution, name="No groups lasso", mic_range=mic_range, source_range=source_range, sparsity_range=sparsity_range, seeds=seeds, vmin=vmin, vmax=vmax)

### Complex group

In [ ]:
heatmap_average_wrong_sources(c_group_solution, name="Complex group lasso", mic_range=mic_range, source_range=source_range, sparsity_range=sparsity_range, seeds=seeds)

### Two groups for purely real or purely imaginary solutions

In [ ]:
heatmap_average_wrong_sources(rigroup_solution, name="Real/Imaginary group lasso", mic_range=mic_range, source_range=source_range, sparsity_range=sparsity_range, seeds=seeds, vmin=vmin, vmax=vmax)

# Broadband setup to group on frequencies

In [ ]:
frequencies = [[100, 200, 300], [300, 600], [400], [100, 500]]
# frequencies = frequencies[0:2]
num_sources = len(frequencies)
num_mics = np.max([num_sources-1, 1])
s_sparse = 2
seed = 1
print(frequencies, num_sources, num_mics)
sim = run_simulation(num_mics = num_mics, num_sources=num_sources, s_sparse=s_sparse, seed=seed, frequencies=frequencies, sampling_rate_factor=2)
plot_overview(sim)
plot_matrices([sim.X, sim.Y, sim.C[:,:,1]], titles=["True X", "Measurements Y", "Mixing Matrix C at f=1"])

# Group on frequencies

In [ ]:
# Convert to block matrix and augmented real form and all that
from cs_priors.solvers.moe_group_lasso import create_frequency_groups, tensor_to_block_matrix, matrix_to_block_vector
frequency_groups = create_frequency_groups(num_sources=num_sources, num_frequencies=sim.C.shape[2])
print(frequency_groups)
# check the new matrix
A_block = tensor_to_block_matrix(sim.C)
Y_block = matrix_to_block_vector(sim.Y)
X_true_block = matrix_to_block_vector(sim.X)
print(len(from_real_augmented(frequency_groups)), len(X_true_block))
plot_matrices([from_real_augmented(frequency_groups), X_true_block], titles=["Frequency Groups (complex)", "True X Block"], show_values=True)
plot_matrices([sim.Y, Y_block], titles=["Y", "Y Block"], show_values=True)

plot_matrices([A_block, sim.C[:,0,:]], titles=["Block Matrix A", "Mixing Matrix A[:,0,:] at column 0"], show_values=False)


In [ ]:
X0 = np.linalg.pinv(A_block) @ Y_block
group_reg = 1e-4 * num_mics
max_iter = 100000

model = GroupLasso(
    groups=frequency_groups, group_reg=group_reg, n_iter=max_iter, supress_warning=True
)
X_gl = from_real_augmented(model.fit(to_real_augmented(A_block), to_real_augmented(Y_block)).coef_.reshape(-1, 1))

plot_equation(X_true_block, X_gl, X0, titles=["True X Block", "Group Lasso X Block", "Pseudoinverse X0 Block"])


# Andre grupper uten frekvenser

In [ ]:
X_nogroup = no_group_solution(Y_block, A_block)
X_cgroup = c_group_solution(Y_block, A_block)
X_rigroup = rigroup_solution(Y_block, A_block)

plot_matrices(
    [X_true_block, X_nogroup, X_cgroup, X_rigroup, X_gl], 
    titles=["True X Block", "No Group Lasso", "Complex pair Group Lasso", "Pure Real/Imag Group Lasso", "Frequency and complex pair Group Lasso"]
)

# Kvantifiser hvor godt grupperinger hjelper på blokkstrukturen
Trekk tilfeldig utvalg av frekvenser for hver kilde

In [ ]:
all_frequencies = []
for source in range(num_sources):
    count = np.random.randint(1, 10)
    frequencies = []
    for _ in range(count):
        frequencies.append(np.random.randint(100, 500))
    all_frequencies.append(frequencies)
print(all_frequencies)

### Lag heatmap over hvor godt metodene gjør det med et tilfeldig uttrekk av frekvenser i neste notatbok -> ``Block_group.ipynb``

In [ ]:
source_range = [10]
mic_range = [2,3,7,10]
sparsity_range = [1,2,3]
seeds = 10
vmin = 0
vmax = source_range[0]//2
freq_range = (100, 500)

heatmap_average_wrong_sources(
    method1=rigroup_solution, 
    name="pure real/imaginary lasso", 
    mic_range=mic_range, 
    source_range=source_range, 
    sparsity_range=sparsity_range, 
    seeds=seeds, 
    vmin=vmin, 
    vmax=vmax,
    frequency_idx=None,
    freq_interval=freq_range,
)
